# 06f — Frequency-preserving order-shuffle null (G=28)

Kickoff **E** of the E→G→F arc (`.claude/plans/coordination-EFG-roadmap.md`).
Quantifies *how much group-discriminative information lives in symbol **order** vs **frequency***,
with the higher-leverage probe the prior tests (06e; `RESULTS_HANDOFF §12.7/§13.4`) lacked: a
**frequency-preserving order-shuffle null**.

For the best order-aware classifier, score it on the *intact* sequences (`AUC_intact`) and on many
*within-sequence shuffles* that preserve each sequence's symbol multiset **exactly** (`AUC_shuffled`).
Report

> **ΔAUC = AUC_intact − mean(AUC_shuffled)**, with a 95% CI from the shuffle distribution.

Because the shuffle holds *frequency* fixed, ΔAUC isolates *order alone*; and because the **same
pipeline** hits intact and shuffled data, classifier optimism/overfitting bias cancels in ΔAUC — the
null is its own control (fixing the overfit weakness of the 06e incremental-bigram probe).

**Two nulls** (both per-sequence ⇒ the unigram histogram is exactly invariant):
- **`token`** — uniform within-sequence permutation; destroys order **and** dwell. ΔAUC = *all*
  structure beyond bare frequency (transitions + dwell).
- **`runlength`** — permute the *order of runs* (RLE), preserving the per-symbol dwell-time
  distribution; destroys only run sequencing. ΔAUC = *pure sequencing*, frequency **and** dwell held.

The two coincide at segment-level (runs length-1) and diverge at embedding-level (long dwell runs).

### Pre-registration (stated before any result)
> **This is a measurement, not a goal.** For each comparison × feature × null:
> **order signal present iff the ΔAUC 95% CI lower bound `ci_low > 0`** (≡ `order_helps`). The
> expected, paper-hardening result is `ci_low ≤ 0` everywhere (order carries no group signal beyond
> frequency). Any `ci_low > 0` is flagged for a pre-registered powered follow-up — no post-hoc tuning.

In [1]:
%load_ext autoreload
%autoreload 2
import os
os.environ.setdefault('NUMBA_THREADING_LAYER', 'workqueue')  # fork-safe rTWE under nbconvert
os.environ.setdefault('MPLBACKEND', 'agg')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
from smartflat.utils.utils_io import get_data_root
from smartflat.utils.utils import upsample_sequence
from smartflat.features.symbolic_barycenter import vocab
from smartflat.features.symbolic_barycenter import order_evaluation as OE
plt.rcParams['figure.dpi'] = 110

OUT = os.path.join(get_data_root(), 'outputs', 'symbolic_barycenter', 'g28')
EXP = os.path.join(OUT, 'experiments')
os.makedirs(EXP, exist_ok=True)

# CV / null budget (shared across all calls). 06e-scale runtime.
N_REPEATS, N_FOLDS, N_SHUFFLES, RS = 2, 5, 100, 42
print('output dir:', EXP)

output dir: /home/perochon/data-gold-final/outputs/symbolic_barycenter/g28/experiments


In [2]:
# Canonical G=28 cohort + ground cost via the shared loaders (06c Cells 2-3).
df, X_embed, labels = vocab.load_g28_cohort(rep='int_cat_segm_embedding_labels')   # ragged, L~5162
_,  X_seg_ragged, _ = vocab.load_g28_cohort(rep='int_cat_segments_labels')          # ragged, L~177
D_G_cat = vocab.build_g28_ground_cost()
G28 = D_G_cat.shape[0]

# segment-level upsampled to the cohort median segment count (mirrors 06e)
L_seg = int(np.median([len(s) for s in X_seg_ragged]))
X_seg = [upsample_sequence(np.asarray(s), L_seg).astype(int) for s in X_seg_ragged]

print('cohort:', df.groupby('pathologie').size().to_dict(), '| G =', G28)
print('embedding L: median %d  (min %d, max %d)' % (
    int(np.median([len(s) for s in X_embed])), min(map(len, X_embed)), max(map(len, X_embed))))
print('segment   L: %d  (upsampled from median %d)' % (L_seg, L_seg))

cohort: {'HEALTHY': 24, 'RIL': 37, 'TBI': 59} | G = 28
embedding L: median 5170  (min 2292, max 9570)
segment   L: 176  (upsampled from median 176)


## (1) Segment-level (`int_cat_segments_labels`, L≈median) — the action grammar

At segment-level, runs are length-1, so `token` ≈ `runlength` (only one null exercised). Two
order-aware features: frame-level bigram `transition` and the dwell-invariant `run_transition`.

In [3]:
seg_runs = []
for feat in ['transition', 'run_transition']:
    r = OE.order_information(X_seg, labels, G28, feature=feat, shuffle='token',
                             n_repeats=N_REPEATS, n_folds=N_FOLDS, n_shuffles=N_SHUFFLES,
                             random_state=RS)
    r.insert(0, 'rep', 'segment')
    seg_runs.append(r)
seg = pd.concat(seg_runs, ignore_index=True)
display(seg.round(3))

,rep,comparison,feature,shuffle,classifier,n_shuffles,auc_intact,auc_null_mean,delta_auc,ci_low,ci_high,p_perm,order_helps
0,segment,HEALTHY_vs_RIL,transition,token,logreg,100,0.741,0.753,-0.013,-0.106,0.099,0.614,False
1,segment,RIL_vs_TBI,transition,token,logreg,100,0.651,0.632,0.019,-0.089,0.141,0.366,False
2,segment,CONTROL_vs_PATIENT,transition,token,logreg,100,0.694,0.651,0.043,-0.069,0.156,0.277,False
3,segment,HEALTHY_vs_RIL,run_transition,token,logreg,100,0.733,0.749,-0.017,-0.122,0.094,0.634,False
4,segment,RIL_vs_TBI,run_transition,token,logreg,100,0.616,0.632,-0.016,-0.125,0.112,0.604,False
5,segment,CONTROL_vs_PATIENT,run_transition,token,logreg,100,0.663,0.649,0.014,-0.083,0.125,0.426,False


## (2) Embedding-level (`int_cat_segm_embedding_labels`, L≈5162) — where the two nulls separate

- `transition × token`     → all structure beyond bare frequency (transitions **+** dwell)
- `transition × runlength` → pure sequencing via frame bigrams (dwell held fixed)
- `run_transition × runlength` → dwell-invariant run-grammar sequencing (the cleanest sequencing probe)

In [4]:
configs = [('transition', 'token'), ('transition', 'runlength'), ('run_transition', 'runlength')]
emb_runs = []
for feat, sh in configs:
    r = OE.order_information(X_embed, labels, G28, feature=feat, shuffle=sh,
                             n_repeats=N_REPEATS, n_folds=N_FOLDS, n_shuffles=N_SHUFFLES,
                             random_state=RS)
    r.insert(0, 'rep', 'embedding')
    emb_runs.append(r)
emb = pd.concat(emb_runs, ignore_index=True)
display(emb.round(3))

,rep,comparison,feature,shuffle,classifier,n_shuffles,auc_intact,auc_null_mean,delta_auc,ci_low,ci_high,p_perm,order_helps
0,embedding,HEALTHY_vs_RIL,transition,token,logreg,100,0.821,0.848,-0.028,-0.066,0.021,0.931,False
1,embedding,RIL_vs_TBI,transition,token,logreg,100,0.607,0.709,-0.103,-0.159,-0.030,0.990,False
2,embedding,CONTROL_vs_PATIENT,transition,token,logreg,100,0.741,0.707,0.034,-0.021,0.102,0.149,False
3,embedding,HEALTHY_vs_RIL,transition,runlength,logreg,100,0.821,0.757,0.064,-0.040,0.179,0.129,False
4,embedding,RIL_vs_TBI,transition,runlength,logreg,100,0.607,0.638,-0.032,-0.118,0.075,0.782,False
5,embedding,CONTROL_vs_PATIENT,transition,runlength,logreg,100,0.741,0.663,0.079,-0.039,0.189,0.109,False
6,embedding,HEALTHY_vs_RIL,run_transition,runlength,logreg,100,0.773,0.750,0.023,-0.072,0.119,0.376,False
7,embedding,RIL_vs_TBI,run_transition,runlength,logreg,100,0.641,0.641,-0.000,-0.088,0.088,0.495,False
8,embedding,CONTROL_vs_PATIENT,run_transition,runlength,logreg,100,0.647,0.641,0.006,-0.128,0.134,0.455,False


In [5]:
# persist the full table for §15 / downstream sessions
allres = pd.concat([seg, emb], ignore_index=True)
allres.to_csv(os.path.join(EXP, 'order_shuffle_null_deltaAUC.csv'), index=False)
print('saved', os.path.join(EXP, 'order_shuffle_null_deltaAUC.csv'), '| rows', len(allres))
display(allres[['rep', 'feature', 'shuffle', 'comparison', 'auc_intact', 'auc_null_mean',
                'delta_auc', 'ci_low', 'ci_high', 'p_perm', 'order_helps']].round(3))

saved /home/perochon/data-gold-final/outputs/symbolic_barycenter/g28/experiments/order_shuffle_null_deltaAUC.csv | rows 15


,rep,feature,shuffle,comparison,auc_intact,auc_null_mean,delta_auc,ci_low,ci_high,p_perm,order_helps
0,segment,transition,token,HEALTHY_vs_RIL,0.741,0.753,-0.013,-0.106,0.099,0.614,False
1,segment,transition,token,RIL_vs_TBI,0.651,0.632,0.019,-0.089,0.141,0.366,False
2,segment,transition,token,CONTROL_vs_PATIENT,0.694,0.651,0.043,-0.069,0.156,0.277,False
3,segment,run_transition,token,HEALTHY_vs_RIL,0.733,0.749,-0.017,-0.122,0.094,0.634,False
4,segment,run_transition,token,RIL_vs_TBI,0.616,0.632,-0.016,-0.125,0.112,0.604,False
5,segment,run_transition,token,CONTROL_vs_PATIENT,0.663,0.649,0.014,-0.083,0.125,0.426,False
6,embedding,transition,token,HEALTHY_vs_RIL,0.821,0.848,-0.028,-0.066,0.021,0.931,False
7,embedding,transition,token,RIL_vs_TBI,0.607,0.709,-0.103,-0.159,-0.030,0.990,False
8,embedding,transition,token,CONTROL_vs_PATIENT,0.741,0.707,0.034,-0.021,0.102,0.149,False
9,embedding,transition,runlength,HEALTHY_vs_RIL,0.821,0.757,0.064,-0.040,0.179,0.129,False


In [6]:
# ΔAUC with 95% CI error bars; green = CI excludes 0 (order signal), grey otherwise
fig, ax = plt.subplots(figsize=(13, 5))
lab = (allres['rep'].str[:3] + ':' + allres['feature'].str.replace('_', '') + '/'
       + allres['shuffle'].str[:3] + '\n' + allres['comparison'])
x = np.arange(len(allres))
yerr = np.vstack([allres['delta_auc'] - allres['ci_low'],
                  allres['ci_high'] - allres['delta_auc']])
colors = ['C2' if h else '0.6' for h in allres['order_helps']]
ax.bar(x, allres['delta_auc'], yerr=yerr, capsize=3, color=colors)
ax.axhline(0, color='k', lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(lab, fontsize=7, rotation=90)
ax.set_ylabel(r'$\Delta$AUC  (intact $-$ shuffled)')
ax.set_title('Order-shuffle null: group-discriminative information in order vs frequency (G=28)\n'
             'green = 95% CI excludes 0 (order signal present); grey = CI brackets 0')
plt.tight_layout()
plt.savefig(os.path.join(EXP, 'order_shuffle_null_deltaAUC.png'), dpi=130)
plt.show()

/tmp/ipykernel_2034776/1377643003.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# data-driven verdict (no post-hoc tuning)
pos = allres[allres['order_helps']]
print('=== Order-shuffle-null verdict (ci_low > 0 ⇒ order signal beyond frequency) ===')
for _, r in allres.iterrows():
    flag = 'ORDER SIGNAL' if r['order_helps'] else 'no signal'
    print('  %-9s %-14s %-9s %-19s ΔAUC=%+.3f [%+.3f, %+.3f]  p=%.3f  -> %s' % (
        r['rep'], r['feature'], r['shuffle'], r['comparison'],
        r['delta_auc'], r['ci_low'], r['ci_high'], r['p_perm'], flag))
print('\\n%d / %d (comparison × feature × null) cells show an order signal (CI excludes 0).'
      % (len(pos), len(allres)))
if len(pos):
    print('FLAGGED for pre-registered follow-up:')
    display(pos[['rep', 'feature', 'shuffle', 'comparison', 'delta_auc', 'ci_low', 'ci_high', 'p_perm']].round(3))
else:
    print('VERDICT: order carries no group-discriminative signal beyond frequency, '
          'now via the stronger shuffle-null probe — consistent with 06e and hardening the paper.')

=== Order-shuffle-null verdict (ci_low > 0 ⇒ order signal beyond frequency) ===
  segment   transition     token     HEALTHY_vs_RIL      ΔAUC=-0.013 [-0.106, +0.099]  p=0.614  -> no signal
  segment   transition     token     RIL_vs_TBI          ΔAUC=+0.019 [-0.089, +0.141]  p=0.366  -> no signal
  segment   transition     token     CONTROL_vs_PATIENT  ΔAUC=+0.043 [-0.069, +0.156]  p=0.277  -> no signal
  segment   run_transition token     HEALTHY_vs_RIL      ΔAUC=-0.017 [-0.122, +0.094]  p=0.634  -> no signal
  segment   run_transition token     RIL_vs_TBI          ΔAUC=-0.016 [-0.125, +0.112]  p=0.604  -> no signal
  segment   run_transition token     CONTROL_vs_PATIENT  ΔAUC=+0.014 [-0.083, +0.125]  p=0.426  -> no signal
  embedding transition     token     HEALTHY_vs_RIL      ΔAUC=-0.028 [-0.066, +0.021]  p=0.931  -> no signal
  embedding transition     token     RIL_vs_TBI          ΔAUC=-0.103 [-0.159, -0.030]  p=0.990  -> no signal
  embedding transition     token     CONTROL_vs_

## Honest conclusion

This notebook applies one pre-registered decision rule — **order signal present iff the ΔAUC 95% CI
excludes 0** — with no post-hoc tuning, on the faithful G=28 cohort, at both representations and both
frequency-preserving nulls. The verdict cell above reports the numbers.

The expected, paper-hardening outcome is that every ΔAUC CI brackets 0: *the discriminative signal is
in **which** actions occur and **how often**, not **in what order***. Because the shuffle preserves
each sequence's histogram exactly and the same pipeline scores intact and shuffled data, a null ΔAUC
is a clean isolation of "no order signal beyond frequency" — strictly stronger than 06e's incremental
probe (which overfit the 784-dim bigram block). Any cell with `ci_low > 0` is **not** declared a win
here; it is flagged for a powered, pre-registered follow-up.

Reusable API shipped in `smartflat/features/symbolic_barycenter/order_evaluation.py`
(`token_shuffle`, `runlength_shuffle`, `order_shuffle_null`, `run_transition_features`,
`order_information`) — Kickoff F scores barycenters on whether they preserve whatever order signal is
found here; Kickoff G's behavioral metrics are order-aware features evaluated through this same null.